<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-10d-columnar-sql-dbt-style-models-and-dashboards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 10 (continued) - Where the features come from: columnar analytics, dbt-style models, and the dashboard on top

A model service is only as good as the table it is fed. Data scientists are often asked to work with an **analytical data platform** (ClickHouse, Snowflake, BigQuery), to build **transformation models** the way dbt does, and to put the result in front of people through a BI tool (Superset, Tableau, Looker). This lab teaches the transferable ideas on a laptop-sized stand-in, **DuckDB**, using three million real taxi trips: why column stores are fast, how to write feature queries with window functions, how a dbt-style project is ordered and tested, and what "bytes scanned" means for your bill.

**Optional advanced-discussion lab** for Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Runs on the free Colab CPU in a couple of minutes (it downloads a 50 MB file).

*Original code written for this course. Data: NYC Taxi & Limousine Commission Yellow Taxi trip records, January 2024, published by the NYC TLC as open data. DuckDB is MIT-licensed. dbt, ClickHouse, Snowflake and BigQuery are described from their public documentation; this lab does not run them.*

In [ ]:
%pip install -q duckdb pandas matplotlib pyarrow

In [ ]:
import os, time, sqlite3, urllib.request, tempfile
import duckdb, pandas as pd, numpy as np
import pyarrow.parquet as pq

URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
PATH = os.path.join(tempfile.gettempdir(), "yellow_tripdata_2024-01.parquet")
if not os.path.exists(PATH): urllib.request.urlretrieve(URL, PATH)
con = duckdb.connect()
print(f"{os.path.getsize(PATH)/1e6:.1f} MB on disk |", con.sql(f"SELECT count(*) AS trips FROM '{PATH}'").fetchone()[0], "trips |", len(pq.ParquetFile(PATH).schema.names), "columns")
con.sql(f"SELECT * FROM '{PATH}' LIMIT 3").df()

## Part 1 - Why column stores are fast for analytics

A **row store** (SQLite, Postgres, MySQL) keeps each trip's 19 fields together: ideal for "fetch or update trip 12345". A **column store** (DuckDB, ClickHouse, BigQuery, Snowflake) keeps each field together across all trips: ideal for "average fare by hour over millions of trips", because the query reads only the two columns it needs and compresses each column well. We load the same 1,000,000 trips into SQLite (row store) and DuckDB (column store) and ask the same question.

In [ ]:
N = 1_000_000
df = con.sql(f"SELECT * FROM '{PATH}' LIMIT {N}").df()
df["tpep_pickup_datetime"] = df["tpep_pickup_datetime"].astype(str); df["tpep_dropoff_datetime"] = df["tpep_dropoff_datetime"].astype(str)

sq = sqlite3.connect(":memory:"); t0 = time.time()
df.to_sql("trips", sq, index=False, if_exists="replace", chunksize=100_000)
print(f"loaded {len(df):,} rows into SQLite (row store) in {time.time()-t0:.1f}s")
col = duckdb.connect(); col.register("df_view", df); col.sql("CREATE TABLE trips AS SELECT * FROM df_view")
print("loaded the same rows into DuckDB (column store)")

Q_SQLITE = "SELECT CAST(substr(tpep_pickup_datetime, 12, 2) AS INTEGER) AS hr, avg(fare_amount) AS avg_fare, count(*) AS n FROM trips GROUP BY hr ORDER BY hr"
Q_DUCK   = "SELECT CAST(substr(tpep_pickup_datetime, 12, 2) AS INTEGER) AS hr, avg(fare_amount) AS avg_fare, count(*) AS n FROM trips GROUP BY hr ORDER BY hr"

def best(f, reps=3):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); out = f(); ts.append(time.perf_counter() - t0)
    return min(ts), out
t_row, r_row = best(lambda: sq.execute(Q_SQLITE).fetchall())
t_col, r_col = best(lambda: col.sql(Q_DUCK).fetchall())
print(f"\nsame query, same 1,000,000 rows:  row store {t_row*1000:8.1f} ms | column store {t_col*1000:7.1f} ms | speed-up x{t_row/t_col:.0f}")
print("same answer:", all(abs(a[1]-b[1]) < 1e-9 and a[2] == b[2] for a, b in zip(r_row, r_col)))

**Read it.** The column store answers in a few milliseconds what the row store needs a large fraction of a second for, on identical data and an identical answer. (SQLite is a fine database; it is simply built for a different job.) The ratio, not the milliseconds, is the lesson, and it grows with table width and size.

## Part 2 - What you are actually charged for: bytes scanned

BigQuery's on-demand pricing is based on the **bytes a query reads**, and Snowflake charges for warehouse **time**, which also tracks how much data is read. Both reward the same habit: *select only the columns you need, and filter on the partition or clustering key*. We can measure the first half directly from the Parquet file's column sizes.

In [ ]:
pf = pq.ParquetFile(PATH); md_ = pf.metadata
size = {}
for rg in range(md_.num_row_groups):
    for c in range(md_.num_columns):
        cc = md_.row_group(rg).column(c); size[cc.path_in_schema] = size.get(cc.path_in_schema, 0) + cc.total_compressed_size
total = sum(size.values())
def share(cols): return sum(size[c] for c in cols) / total
print(f"file column data: {total/1e6:.1f} MB over {len(size)} columns")
print(f"{'query reads these columns':56}{'share of bytes':>16}")
for label, cols in [("SELECT * (everything)", list(size)),
                    ("tpep_pickup_datetime, fare_amount", ["tpep_pickup_datetime", "fare_amount"]),
                    ("PULocationID, total_amount", ["PULocationID", "total_amount"]),
                    ("fare_amount only", ["fare_amount"])]:
    print(f"{label:56}{share(cols):>15.1%}")
print("\nlargest columns:", ", ".join(f"{k} {v/total:.0%}" for k, v in sorted(size.items(), key=lambda kv: -kv[1])[:3]))

**Read it.** A two-column query touches a small share of the file; `SELECT *` touches all of it. On a pay-per-byte platform that is the difference between a cheap query and an expensive one, and `SELECT *` on a wide table is the classic cost mistake. The second half of the habit, **partitioning and clustering**, works the same way at a coarser grain: a table partitioned by date lets a one-day query skip every other day's files entirely (BigQuery partitions and clusters; Snowflake keeps data in micro-partitions with optional clustering keys; ClickHouse sorts data by the table's `ORDER BY` key and keeps a sparse primary index over it, so range filters on the leading key columns skip most of the data).

## Part 3 - Feature queries: CTEs and window functions

Features for a model are mostly aggregations over time. Window functions compute them without self-joins. Below: per pickup zone, trips per hour, the same zone's previous-hour count (a **lag feature**) and a 3-hour rolling average, then a ranking to find each zone's busiest hour.

In [ ]:
FEATURE_SQL = f"""
WITH trips AS (
  SELECT PULocationID AS zone, date_trunc('hour', tpep_pickup_datetime) AS hr, fare_amount
  FROM '{PATH}'
  WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01' AND tpep_pickup_datetime < TIMESTAMP '2024-02-01' AND fare_amount > 0
),
hourly AS (
  SELECT zone, hr, count(*) AS n_trips, avg(fare_amount) AS avg_fare FROM trips GROUP BY zone, hr
)
SELECT zone, hr, n_trips, round(avg_fare, 2) AS avg_fare,
       lag(n_trips, 1) OVER w                                   AS trips_prev_hour,
       round(avg(n_trips) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS trips_3h_avg,
       row_number() OVER (PARTITION BY zone ORDER BY n_trips DESC) AS busiest_rank
FROM hourly
WINDOW w AS (PARTITION BY zone ORDER BY hr)
ORDER BY zone, hr
"""
t0 = time.time(); feats = con.sql(FEATURE_SQL).df()
print(f"{len(feats):,} zone-hour rows built straight from the Parquet file in {time.time()-t0:.2f}s")
print(feats[feats.zone == 161].head(6).to_string(index=False))
top = feats[feats.busiest_rank == 1].sort_values("n_trips", ascending=False).head(5)
print("\nbusiest zone-hours in the month:\n", top[["zone", "hr", "n_trips"]].to_string(index=False))

## Part 4 - A dbt-style project: models, dependencies and tests

**dbt** (data build tool, open-source core under Apache-2.0) turns a folder of `SELECT` statements into a tested, ordered set of tables. Each file is a *model*; `{{ ref('other_model') }}` declares a dependency; dbt works out the order, builds the models in your warehouse, and runs *tests* (`not_null`, `unique`, `accepted_values`, `relationships`, or any SQL that must return zero rows). We rebuild the idea in about thirty lines so you can see what the tool automates: staging -> intermediate -> mart, built in dependency order, then tested.

In [ ]:
import re
from graphlib import TopologicalSorter

MODELS = {   # name -> SQL, exactly as you would put it in models/<name>.sql
 "stg_trips": f"""SELECT PULocationID AS zone, tpep_pickup_datetime AS picked_up, fare_amount, trip_distance
                  FROM '{PATH}'
                  WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01' AND tpep_pickup_datetime < TIMESTAMP '2024-02-01'
                    AND fare_amount > 0 AND trip_distance > 0""",
 "int_zone_hour": "SELECT zone, date_trunc('hour', picked_up) AS hr, count(*) AS n_trips, avg(fare_amount) AS avg_fare FROM {{ ref('stg_trips') }} GROUP BY zone, hr",
 "mart_zone_features": """SELECT zone, hr, n_trips, avg_fare,
                          lag(n_trips) OVER (PARTITION BY zone ORDER BY hr) AS trips_prev_hour,
                          hour(hr) AS hour_of_day, dayofweek(hr) AS day_of_week
                        FROM {{ ref('int_zone_hour') }}""",
}
TESTS = {   # model -> list of (test name, SQL that must return ZERO rows)
 "stg_trips": [("fare_amount is positive", "SELECT * FROM {{ ref('stg_trips') }} WHERE fare_amount <= 0"),
               ("zone is not null", "SELECT * FROM {{ ref('stg_trips') }} WHERE zone IS NULL")],
 "mart_zone_features": [("(zone, hr) is unique", "SELECT zone, hr FROM {{ ref('mart_zone_features') }} GROUP BY zone, hr HAVING count(*) > 1"),
                        ("n_trips is positive", "SELECT * FROM {{ ref('mart_zone_features') }} WHERE n_trips <= 0")],
}
REF = re.compile(r"\{\{\s*ref\('([^']+)'\)\s*\}\}")
def render(sql): return REF.sub(lambda m: m.group(1), sql)
deps = {n: set(REF.findall(s)) for n, s in MODELS.items()}
order = list(TopologicalSorter(deps).static_order())
print("build order from the ref() graph:", " -> ".join(order))

db = duckdb.connect()
def dbt_run():
    for name in order:
        t0 = time.time(); db.sql(f"CREATE OR REPLACE TABLE {name} AS {render(MODELS[name])}")
        print(f"  built {name:20} {db.sql(f'SELECT count(*) FROM {name}').fetchone()[0]:>9,} rows  ({time.time()-t0:.1f}s)")
def dbt_test():
    failures = 0
    for model, tests in TESTS.items():
        for tname, sql in tests:
            bad = len(db.sql(render(sql)).fetchall()); failures += bad > 0
            print(f"  {'PASS' if bad == 0 else 'FAIL'}  {model}: {tname}" + (f"  ({bad} offending rows)" if bad else ""))
    return failures
print("dbt run"); dbt_run()
print("dbt test"); print("failed tests:", dbt_test())

**Break it on purpose.** A test suite is only trustworthy if you have seen it fail. We inject a duplicate row into the mart and run the tests again.

In [ ]:
db.sql("INSERT INTO mart_zone_features SELECT * FROM mart_zone_features LIMIT 1")        # simulate a bad join duplicating a row
print("dbt test after the injected duplicate"); print("failed tests:", dbt_test())
dbt_run()                                                                                    # a rebuild restores the model
print("after rebuild: failed tests:", dbt_test())

**What the real tool adds:** a `dbt_project.yml`, `sources` declared separately from models, incremental models (process only new rows), documentation generated from the same files, and adapters for Snowflake, BigQuery, ClickHouse, DuckDB and others (`dbt-snowflake`, `dbt-bigquery`, `dbt-clickhouse`, `dbt-duckdb`). The workflow you have just seen - ordered builds, tests that must return zero rows, run in CI before anything reaches production - is the same.

## Part 5 - The dashboard on top

A BI tool (**Apache Superset**, open source under Apache-2.0; **Tableau**; **Looker**) connects to the warehouse, runs SQL against your marts, and renders charts for people who will never open a notebook. Their value is *shared, governed definitions*: one agreed definition of "trips" or "average fare" so every chart agrees. Your job as the data scientist is the mart underneath it: correct, tested, and cheap to query. Here is the chart such a tool would draw from our mart.

In [ ]:
import matplotlib.pyplot as plt
by_hour = db.sql("SELECT hour_of_day, sum(n_trips) AS trips, sum(avg_fare * n_trips) / sum(n_trips) AS avg_fare FROM mart_zone_features GROUP BY hour_of_day ORDER BY hour_of_day").df()
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.bar(by_hour.hour_of_day, by_hour.trips / 1000, color="#1F3A5F")
ax.set_xlabel("hour of day"); ax.set_ylabel("trips (thousands)"); ax2 = ax.twinx(); ax2.plot(by_hour.hour_of_day, by_hour.avg_fare, color="#C0392B", marker="o"); ax2.set_ylabel("average fare ($)")
ax.set_title("Yellow taxi, January 2024: trips and average fare by hour (from mart_zone_features)"); fig.tight_layout()
plt.show()
print(by_hour.round(2).head(24).to_string(index=False))

## How the platforms differ (from their public documentation)

| | DuckDB (this lab) | ClickHouse | BigQuery | Snowflake |
|---|---|---|---|---|
| What it is | In-process analytical engine (MIT) | Open-source (Apache-2.0) column-oriented DBMS for real-time analytics; also a managed cloud | Serverless managed warehouse | Managed cloud warehouse |
| Data layout | Columnar, vectorised | Columnar; `MergeTree` tables sorted by an `ORDER BY` key with a sparse primary index; partitions | Columnar (Capacitor); partitioning and clustering | Columnar micro-partitions; optional clustering keys |
| You pay for | Your own CPU | Your servers, or the managed service | Bytes scanned (on-demand) or reserved capacity | Warehouse running time plus storage |
| First habit to build | `EXPLAIN` the query | Choose the `ORDER BY` key to match your filters | Select only needed columns; filter on the partition column; use the dry-run estimate | Right-size and auto-suspend the warehouse |

Syntax differs in details (for instance `QUALIFY` exists in DuckDB, BigQuery and Snowflake but not in ClickHouse), so keep SQL standard where you can and read each platform's documentation before relying on a feature.

In [ ]:
col.close(); db.close(); sq.close(); con.close(); print("closed")

## What to hand in

The row-store versus column-store timing table with your own numbers, the bytes-share table, one window-function feature query of your own on a different column, a dbt-style project of three models with at least four tests (show one failing and fixed), and a paragraph saying which platform you would pick for a real-time identification workload and why.